# 4. Rating Prediction (Regression)

In [2]:
import os, sys
try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _HERE = os.getcwd()
for _c in (_HERE, os.path.dirname(_HERE)):
    _p = os.path.join(_c, "app")
    if os.path.isdir(_p):
        sys.path.insert(0, _p); break

import joblib, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score
from text_utils import CLEAN_CSV, MODELS_DIR, probs_to_outputs, clean_review_text

bundle = joblib.load(os.path.join(MODELS_DIR, "review_model.pkl"))
pipe, classes = bundle["pipeline"], np.asarray(bundle["classes"])
df = pd.read_csv(CLEAN_CSV)
X, y = df["review_clean"].astype(str).values, df["Rating"].astype(int).values
print("loaded model trained with C =", bundle["best_C"])

loaded model trained with C = 0.5


In [5]:
X = [str(t) for t in df["review_clean"].tolist()]
y = df["Rating"].astype(int).to_numpy()
print(type(X), len(X))   # should print: <class 'list'> 587 (or close)

<class 'list'> 587


In [6]:
# Honest CV: vectorizer + model refit inside every fold (old code leaked)
cv_proba = cross_val_predict(pipe, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42),
                             method="predict_proba", n_jobs=1)   # ← was -1
ev = (cv_proba * classes.astype(float)).sum(axis=1)
mse = ((y - ev) ** 2).mean()
print("Rating RMSE : %.3f" % np.sqrt(mse))
print("Rating MAE  : %.3f" % mean_absolute_error(y, ev))
print("Rating R2   : %.3f" % r2_score(y, ev))
print("Predicted range: %.2f - %.2f  (old Ridge was stuck at 2.9-4.5)" % (ev.min(), ev.max()))

Rating RMSE : 0.921
Rating MAE  : 0.759
Rating R2   : 0.513
Predicted range: 1.29 - 4.74  (old Ridge was stuck at 2.9-4.5)


In [7]:
# The six screenshots — rating must now move with sentiment
tests = [
    "the food was amazing and the ambience is too good",
    "the food was amazing and the ambience is too good but the behaviour of staff was very rude",
    "the food was not tasty and the ambience is also not good but the behaviour of staff was very rude",
    "the food was not tasty",
    "the food was not tasty and the service was also slow",
    "the food was not amazing the ambience was also not good and the staff behaviour is soo bad",
]
for t in tests:
    r, s, _ = probs_to_outputs(pipe.predict_proba([clean_review_text(t)])[0], classes)
    print(f"{s:8s} {r:.1f}  | {t[:65]}")

positive 4.4  | the food was amazing and the ambience is too good
positive 3.6  | the food was amazing and the ambience is too good but the behavio
negative 1.8  | the food was not tasty and the ambience is also not good but the 
negative 2.0  | the food was not tasty
negative 2.0  | the food was not tasty and the service was also slow
negative 1.7  | the food was not amazing the ambience was also not good and the s


### Load the saved TF-IDF vectorizer

### Train the regressor

### Compare predictors vs actual ratings

### Save the model